Here, I verify the correctness of the values of the expectation value and matrix elements, taking into account that the Hamiltonian is being modified, and that the extended swap test is being used.

This is basically a more comprehensive version of `results_measurement1.ipynb`. 

In [ ]:
import numpy as np
import pickle
import scipy.sparse
from utils_ferm import (
    op_action_tz
)
from utils_CSF_and_UCSF import (
    orthogonal_transform_obt_tbt,
    obt_phys_spatial_to_spin,
    tbt_phys_spatial_to_spin,
    make_short_H_ferm_op
)
from utils_basic import shift_hamiltonian_qubits
from openfermion import (
    FermionOperator,
    jordan_wigner,
    QubitOperator,
    get_sparse_operator,
    normal_ordered
)

from utils_states import (
    somos_to_seniority_config,
    convert_TZ_format_to_sparse_format,
    variance_of_decomp,
    create_composite_state,
    create_composite_state_prepended,
    expectation,
    matrix_element,
    compress_state,
    convert_dense_format_to_sparse_format,
    variance_of_operator,
    variance_of_general_operator
    
)
from utils_hamiltonian_qubit import (
    process_qubit_hamiltonian_to_remove_irrelevant_terms,
    process_qubit_hamiltonian_to_project_onto_symmetric_subspace
)
from utils_hamiltonian_ferm import (
    process_fermionic_hamiltonian_to_remove_irrelevant_terms
)
from utils_partitioning import (
    sorted_insertion_decomposition,
    augment_decomp_with_pauli_x,
    augment_decomp_with_pauli_x_plus_i_pauli_y
)

filename = 'data/old_data1/h2o_data.dump'

with open(filename, 'rb') as f:
    (
    list_CSF,
    list_list_ia_CSF,
    list_list_theta_CSF,
    list_sym_CSF_vec,
    list_UCSF_tz,
    tz_states,
    somos_list,
    psi_GS_UCSF_smik,
    list_orb_rot,
    x_orbrot,
    Enuc,
    obt_spatial,
    tbt_spatial
    ) = pickle.load(f)

#Rotate orbitals 
if len(list_orb_rot) != 0:
    obt, tbt = orthogonal_transform_obt_tbt(x_orbrot,list_orb_rot,obt_spatial,tbt_spatial)
else:
    obt = obt_phys_spatial_to_spin(obt_spatial)
    tbt = tbt_phys_spatial_to_spin(tbt_spatial)


In [ ]:
Nqubits = obt.shape[0]
dim     = 2 ** Nqubits
Norb    = Nqubits // 2
Nstates = len(tz_states)

# generate Hamiltonian and obtain its QWC decomposition, for both expectation values and matrix elements

Hferm          = make_short_H_ferm_op(0, obt, tbt)
Hqub           = jordan_wigner(Hferm)
qubit_constant = Hqub.constant
Hqub          -= qubit_constant
Hqub.compress()

# variance_upper_bound = 0
# for _, coef in Hqub.terms.items():
#     variance_upper_bound += np.abs(coef)
# variance_upper_bound *= variance_upper_bound

# print(f"Upper bound of variance : {variance_upper_bound}\n")


qwc_decomp      = sorted_insertion_decomposition(Hqub, 'qwc')
qwc_decomp_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp, Nqubits)

# generate states and associated state information

configs      = [somos_to_seniority_config(somo, Norb) for somo in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j:

            # load the states 
            bra               = statevectors[i]
            bra_tz            = tz_states[i]
            bra_somos         = somos_list[i]
            bra_config        = configs[i]

            ket               = statevectors[j]
            ket_tz            = tz_states[j]
            ket_somos         = somos_list[j]
            ket_config        = configs[j]

            bra_ket_composite = create_composite_state(bra, ket, Nqubits)

            # process the Hamiltonian for measurements
            Hferm_processed        = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)
            Hqub_temp              = jordan_wigner(Hferm_processed)
            bra_ket_qubit_constant = Hqub_temp.constant
            Hqub_temp             -= bra_ket_qubit_constant
            Hqub_temp.compress()
            Hqub_processed         = process_qubit_hamiltonian_to_remove_irrelevant_terms(Hqub_temp, Nqubits, bra_config, ket_config)
            Hqub_tapered           = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(Hqub_temp, Nqubits, bra_config, ket_config)

            # obtain Hamiltonian fragments
            qwc_decomp_processed      = sorted_insertion_decomposition(Hqub_processed, 'qwc')
            qwc_decomp_processed_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp_processed, Nqubits)

            # # do a bunch of correctness checks

            # # first: should get the same matrix element no matter how Hamiltonian is processed
            # matrix_element1 = np.round((bra @ get_sparse_operator(Hferm, Nqubits)                                   @ ket.T)[0,0] , 8)
            # matrix_element2 = np.round((bra @ get_sparse_operator(Hferm_processed, Nqubits)                         @ ket.T)[0,0] , 8)
            # matrix_element3 = np.round((bra @ get_sparse_operator(Hqub + qubit_constant, Nqubits)                   @ ket.T)[0,0] , 8)
            # matrix_element4 = np.round((bra @ get_sparse_operator(Hqub_processed + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0] , 8)

            # assert matrix_element1 == matrix_element2
            # assert matrix_element1 == matrix_element3
            # assert matrix_element1 == matrix_element4
            # assert matrix_element2 == matrix_element3
            # assert matrix_element2 == matrix_element4
            # assert matrix_element3 == matrix_element4
                
            # # second: should get the same matrix element from the fragments, including with swap test
            # H_from_fragments           = sum(qwc_decomp_processed)
            # H_tensor_1q_from_fragments = sum(qwc_decomp_processed_swap)

            # assert (Hqub_processed - H_from_fragments) == QubitOperator().zero()
            # assert (Hqub_processed * (QubitOperator(f'X{Nqubits}') + 1j * QubitOperator(f'Y{Nqubits}')) 
            #         - H_tensor_1q_from_fragments) == QubitOperator().zero()


            # matrix_element5 = np.round((bra @ get_sparse_operator(H_from_fragments + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0], 8)
            # if i == j:
            #     cur_op = (
            #         H_tensor_1q_from_fragments 
            #         + bra_ket_qubit_constant
            #     )
            #     matrix_element6 = np.round((bra_ket_composite @ 
            #                                 get_sparse_operator(cur_op, Nqubits + 1) @ 
            #                                 bra_ket_composite.T)[0,0], 8)
            # else:
            #     cur_op = (
            #         H_tensor_1q_from_fragments 
            #         + bra_ket_qubit_constant * QubitOperator(f'X{Nqubits}') 
            #         + 1j * bra_ket_qubit_constant * QubitOperator(f'Y{Nqubits}')
            #     )
            #     matrix_element6 = np.round((bra_ket_composite @ 
            #                                 get_sparse_operator(cur_op, Nqubits + 1) @ 
            #                                 bra_ket_composite.T)[0,0], 8)

            # assert matrix_element1 == matrix_element5
            # assert matrix_element1 == matrix_element6
            # assert matrix_element2 == matrix_element5
            # assert matrix_element2 == matrix_element6
            # assert matrix_element3 == matrix_element5
            # assert matrix_element3 == matrix_element6
            # assert matrix_element4 == matrix_element5
            # assert matrix_element4 == matrix_element6
            # assert matrix_element5 == matrix_element6

            # now calculate the variance of the QWC decomposition for both the unprocessed Hamiltonian and the processed Hamiltonian

            if i == j:
                var_og        = variance_of_decomp(qwc_decomp, ket, Nqubits, general=True)
                var_processed = variance_of_decomp(qwc_decomp_processed, ket, Nqubits, general=True)
            
            else:
                var_og        = variance_of_decomp(qwc_decomp_swap, bra_ket_composite, Nqubits + 1, general=True)
                var_processed = variance_of_decomp(qwc_decomp_processed_swap, bra_ket_composite, Nqubits + 1, general=True)

            results[(i,j)] = (bra_config, ket_config, var_processed, var_og)
            print("{:<2} {:<4} {:<10} {:<15} ".format(i, j, np.real(np.round(var_processed, 4)), np.real(np.round(var_og, 4))))


In [ ]:
variance_list_og          = np.array([list(results.values())[i][3] for i in range(len(results.values()))])
variance_og_neyman        = np.sum(variance_list_og ** (1/2)) ** 2

variance_list_processed   = np.array([list(results.values())[i][2] for i in range(len(results.values()))])
variance_processed_neyman = np.sum(variance_list_processed ** (1/2)) ** 2

print(f'''
    Estimator variance using original Hamiltonian                     : {variance_og_neyman}
    Estimator variance using processed Hamiltonians                   : {variance_processed_neyman}

    Number of shots to chemical accuracy using original Hamiltonian   : {variance_og_neyman / 1e-6}
    Number of shots to chemical accuracy using processed Hamiltonians : {variance_processed_neyman / 1e-6}
''')

Repeat experiment using only non-trivial UCSFs. To determine non-trivial CSFs, look at angles and only use states with at least one angle.

In [ ]:
# print angles
list_list_theta_CSF

In [ ]:
Nqubits = obt.shape[0]
dim     = 2 ** Nqubits
Norb    = Nqubits // 2
Nstates = len(tz_states)

# generate Hamiltonian and obtain its QWC decomposition, for both expectation values and matrix elements

Hferm          = make_short_H_ferm_op(0, obt, tbt)
Hqub           = jordan_wigner(Hferm)
qubit_constant = Hqub.constant
Hqub          -= qubit_constant
Hqub.compress()

# variance_upper_bound = 0
# for _, coef in Hqub.terms.items():
#     variance_upper_bound += np.abs(coef)
# variance_upper_bound *= variance_upper_bound

# print(f"Upper bound of variance : {variance_upper_bound}\n")

qwc_decomp      = sorted_insertion_decomposition(Hqub, 'qwc')
qwc_decomp_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp, Nqubits)

# generate states and associated state information

configs      = [somos_to_seniority_config(somo, Norb) for somo in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j and i < 5 and j < 5:

            # load the states 
            bra               = statevectors[i]
            bra_tz            = tz_states[i]
            bra_somos         = somos_list[i]
            bra_config        = configs[i]

            ket               = statevectors[j]
            ket_tz            = tz_states[j]
            ket_somos         = somos_list[j]
            ket_config        = configs[j]

            bra_ket_composite = create_composite_state(bra, ket, Nqubits)

            # process the Hamiltonian for measurements
            Hferm_processed        = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)
            Hqub_temp              = jordan_wigner(Hferm_processed)
            bra_ket_qubit_constant = Hqub_temp.constant
            Hqub_temp             -= bra_ket_qubit_constant
            Hqub_temp.compress()
            Hqub_processed         = process_qubit_hamiltonian_to_remove_irrelevant_terms(Hqub_temp, Nqubits, bra_config, ket_config)
            Hqub_tapered           = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(Hqub_temp, Nqubits, bra_config, ket_config)

            # obtain Hamiltonian fragments
            qwc_decomp_processed      = sorted_insertion_decomposition(Hqub_processed, 'qwc')
            qwc_decomp_processed_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp_processed, Nqubits)

            # # do a bunch of correctness checks

            # # first: should get the same matrix element no matter how Hamiltonian is processed
            # matrix_element1 = np.round((bra @ get_sparse_operator(Hferm, Nqubits)                                   @ ket.T)[0,0] , 8)
            # matrix_element2 = np.round((bra @ get_sparse_operator(Hferm_processed, Nqubits)                         @ ket.T)[0,0] , 8)
            # matrix_element3 = np.round((bra @ get_sparse_operator(Hqub + qubit_constant, Nqubits)                   @ ket.T)[0,0] , 8)
            # matrix_element4 = np.round((bra @ get_sparse_operator(Hqub_processed + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0] , 8)

            # assert matrix_element1 == matrix_element2
            # assert matrix_element1 == matrix_element3
            # assert matrix_element1 == matrix_element4
            # assert matrix_element2 == matrix_element3
            # assert matrix_element2 == matrix_element4
            # assert matrix_element3 == matrix_element4
                
            # # second: should get the same matrix element from the fragments, including with swap test
            # H_from_fragments           = sum(qwc_decomp_processed)
            # H_tensor_1q_from_fragments = sum(qwc_decomp_processed_swap)

            # assert (Hqub_processed - H_from_fragments) == QubitOperator().zero()
            # assert (Hqub_processed * (QubitOperator(f'X{Nqubits}') + 1j * QubitOperator(f'Y{Nqubits}')) 
            #         - H_tensor_1q_from_fragments) == QubitOperator().zero()


            # matrix_element5 = np.round((bra @ get_sparse_operator(H_from_fragments + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0], 8)
            # if i == j:
            #     cur_op = (
            #         H_tensor_1q_from_fragments 
            #         + bra_ket_qubit_constant
            #     )
            #     matrix_element6 = np.round((bra_ket_composite @ 
            #                                 get_sparse_operator(cur_op, Nqubits + 1) @ 
            #                                 bra_ket_composite.T)[0,0], 8)
            # else:
            #     cur_op = (
            #         H_tensor_1q_from_fragments 
            #         + bra_ket_qubit_constant * QubitOperator(f'X{Nqubits}') 
            #         + 1j * bra_ket_qubit_constant * QubitOperator(f'Y{Nqubits}')
            #     )
            #     matrix_element6 = np.round((bra_ket_composite @ 
            #                                 get_sparse_operator(cur_op, Nqubits + 1) @ 
            #                                 bra_ket_composite.T)[0,0], 8)

            # assert matrix_element1 == matrix_element5
            # assert matrix_element1 == matrix_element6
            # assert matrix_element2 == matrix_element5
            # assert matrix_element2 == matrix_element6
            # assert matrix_element3 == matrix_element5
            # assert matrix_element3 == matrix_element6
            # assert matrix_element4 == matrix_element5
            # assert matrix_element4 == matrix_element6
            # assert matrix_element5 == matrix_element6

            # now calculate the variance of the QWC decomposition for both the unprocessed Hamiltonian and the processed Hamiltonian

            if i == j:
                var_og        = variance_of_decomp(qwc_decomp, ket, Nqubits, general=True)
                var_processed = variance_of_decomp(qwc_decomp_processed, ket, Nqubits, general=True)
            
            else:
                var_og        = variance_of_decomp(qwc_decomp_swap, bra_ket_composite, Nqubits + 1, general=True)
                var_processed = variance_of_decomp(qwc_decomp_processed_swap, bra_ket_composite, Nqubits + 1, general=True)

            results[(i,j)] = (bra_config, ket_config, var_processed, var_og)
            print("{:<2} {:<4} {:<10} {:<15}".format(i, j, np.real(np.round(var_processed, 4)), np.real(np.round(var_og, 4))))


In [ ]:
variance_list_og          = np.array([list(results.values())[i][3] for i in range(len(results.values()))])
variance_og_neyman        = np.sum(variance_list_og ** (1/2)) ** 2

variance_list_processed   = np.array([list(results.values())[i][2] for i in range(len(results.values()))])
variance_processed_neyman = np.sum(variance_list_processed ** (1/2)) ** 2

print(f'''
    Estimator variance using original Hamiltonian                     : {variance_og_neyman}
    Estimator variance using processed Hamiltonians                   : {variance_processed_neyman}

    Number of shots to chemical accuracy using original Hamiltonian   : {variance_og_neyman / 1e-6}
    Number of shots to chemical accuracy using processed Hamiltonians : {variance_processed_neyman / 1e-6}
''')

In [2]:
# verify that I get the same matrix elements using the tapered Hamiltonian as with the untapered Hamiltonian

Nqubits       = obt.shape[0]
dim           = 2 ** Nqubits
Norb          = Nqubits // 2
Nstates       = len(tz_states)
fragment_type = 'fc'

# generate Hamiltonian and obtain its QWC decomposition, for both expectation values and matrix elements

Hferm          = make_short_H_ferm_op(0, obt, tbt)
Hqub           = jordan_wigner(Hferm)
qubit_constant = Hqub.constant
Hqub          -= qubit_constant
Hqub.compress()

# variance_upper_bound = 0
# for _, coef in Hqub.terms.items():
#     variance_upper_bound += np.abs(coef)
# variance_upper_bound *= variance_upper_bound

# print(f"Upper bound of variance : {variance_upper_bound}\n")


qwc_decomp      = sorted_insertion_decomposition(Hqub, fragment_type)
qwc_decomp_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp, Nqubits)

# generate states and associated state information

configs      = [somos_to_seniority_config(somo, Norb) for somo in somos_list]
statevectors = [convert_TZ_format_to_sparse_format(dim, tz_state) for tz_state in tz_states]

results = {}

for i in range(Nstates):
    for j in range(Nstates):
        if i >= j:

            # load the states 
            bra                 = statevectors[i]
            bra_tz              = tz_states[i]
            bra_somos           = somos_list[i]
            bra_config          = configs[i]
            bra_t               = convert_dense_format_to_sparse_format(compress_state(bra.toarray()[0]))

            ket                 = statevectors[j]
            ket_tz              = tz_states[j]
            ket_somos           = somos_list[j]
            ket_config          = configs[j]
            ket_t               = convert_dense_format_to_sparse_format(compress_state(ket.toarray()[0]))

            bra_ket_composite   = create_composite_state(bra, ket, Nqubits)
            bra_ket_composite_t = create_composite_state(bra_t, ket_t, Nqubits // 2)

            # process the Hamiltonian for measurements
            Hferm_processed        = process_fermionic_hamiltonian_to_remove_irrelevant_terms(Hferm, Nqubits, bra, ket_tz)
            Hqub_temp              = jordan_wigner(Hferm_processed)
            bra_ket_qubit_constant = Hqub_temp.constant
            Hqub_temp             -= bra_ket_qubit_constant
            Hqub_temp.compress()
            Hqub_processed         = process_qubit_hamiltonian_to_remove_irrelevant_terms(Hqub_temp, Nqubits, bra_config, ket_config)
            Hqub_tapered           = process_qubit_hamiltonian_to_project_onto_symmetric_subspace(Hqub_temp, Nqubits, bra_config, ket_config)

            tapering_constant      = Hqub_tapered.constant
            Hqub_tapered          -= tapering_constant
            Hqub_tapered.compress()

            # obtain Hamiltonian fragments
            qwc_decomp_processed      = sorted_insertion_decomposition(Hqub_processed, fragment_type)
            qwc_decomp_processed_swap = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp_processed, Nqubits)

            qwc_decomp_tapered        = sorted_insertion_decomposition(Hqub_tapered, fragment_type)
            qwc_decomp_tapered_swap   = augment_decomp_with_pauli_x_plus_i_pauli_y(qwc_decomp_tapered, Nqubits // 2)

            # do a bunch of correctness checks

            # first: should get the same matrix element no matter how Hamiltonian is processed
            matrix_element1 = np.round((bra @ get_sparse_operator(Hferm, Nqubits)                                   @ ket.T)[0,0] , 8)
            matrix_element2 = np.round((bra @ get_sparse_operator(Hferm_processed, Nqubits)                         @ ket.T)[0,0] , 8)
            matrix_element3 = np.round((bra @ get_sparse_operator(Hqub + qubit_constant, Nqubits)                   @ ket.T)[0,0] , 8)
            matrix_element4 = np.round((bra @ get_sparse_operator(Hqub_processed + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0] , 8)

            assert matrix_element1 == matrix_element2
            assert matrix_element1 == matrix_element3
            assert matrix_element1 == matrix_element4
            assert matrix_element2 == matrix_element3
            assert matrix_element2 == matrix_element4
            assert matrix_element3 == matrix_element4
                
            # second: should get the same matrix element from the fragments, including with swap test
            H_from_fragments           = sum(qwc_decomp_processed)
            H_tensor_1q_from_fragments = sum(qwc_decomp_processed_swap)

            assert (Hqub_processed - H_from_fragments) == QubitOperator().zero()
            assert (Hqub_processed * (QubitOperator(f'X{Nqubits}') + 1j * QubitOperator(f'Y{Nqubits}')) 
                    - H_tensor_1q_from_fragments) == QubitOperator().zero()


            matrix_element5 = np.round((bra @ get_sparse_operator(H_from_fragments + bra_ket_qubit_constant, Nqubits) @ ket.T)[0,0], 8)
            if i == j:
                cur_op = (
                    H_tensor_1q_from_fragments 
                    + bra_ket_qubit_constant
                )
                matrix_element6 = np.round((bra_ket_composite @ 
                                            get_sparse_operator(cur_op, Nqubits + 1) @ 
                                            bra_ket_composite.T)[0,0], 8)
            else:
                cur_op = (
                    H_tensor_1q_from_fragments 
                    + bra_ket_qubit_constant * QubitOperator(f'X{Nqubits}') 
                    + 1j * bra_ket_qubit_constant * QubitOperator(f'Y{Nqubits}')
                )
                matrix_element6 = np.round((bra_ket_composite @ 
                                            get_sparse_operator(cur_op, Nqubits + 1) @ 
                                            bra_ket_composite.T)[0,0], 8)

            assert matrix_element1 == matrix_element5
            assert matrix_element1 == matrix_element6
            assert matrix_element2 == matrix_element5
            assert matrix_element2 == matrix_element6
            assert matrix_element3 == matrix_element5
            assert matrix_element3 == matrix_element6
            assert matrix_element4 == matrix_element5
            assert matrix_element4 == matrix_element6
            assert matrix_element5 == matrix_element6

            # third: should get the same matrix element using the tapered Hamiltonian and compressed state
            #        and using the swap test on the tapered state
            #        and by decomposing the tapered Hamiltonian into QWC fragments
            matrix_element7 = np.round((bra_t @ get_sparse_operator(Hqub_tapered+bra_ket_qubit_constant+tapering_constant, Nqubits // 2) @ ket_t.T)[0,0], 8)
            matrix_element8 = np.round(
                (bra_ket_composite_t @
                get_sparse_operator((Hqub_tapered+bra_ket_qubit_constant+tapering_constant) * (QubitOperator(f'X{Nqubits//2}')+1j*QubitOperator(f'Y{Nqubits//2}')), Nqubits//2+1) @
                bra_ket_composite_t.T)[0,0], 8
            )
            
            Hqub_tapered_from_fragments           = sum(qwc_decomp_tapered)
            Hqub_tapered_tensor_1q_from_fragments = sum(qwc_decomp_tapered_swap)

            assert (Hqub_tapered - Hqub_tapered_from_fragments) == QubitOperator().zero()
            assert (Hqub_tapered * (QubitOperator(f'X{Nqubits//2}')+1j*QubitOperator(f'Y{Nqubits//2}')) - Hqub_tapered_tensor_1q_from_fragments) == QubitOperator().zero()

            assert matrix_element1 == matrix_element7
            assert matrix_element2 == matrix_element7
            assert matrix_element3 == matrix_element7
            assert matrix_element4 == matrix_element7
            assert matrix_element5 == matrix_element7
            assert matrix_element6 == matrix_element7

            assert matrix_element1 == matrix_element8
            assert matrix_element2 == matrix_element8
            assert matrix_element3 == matrix_element8
            assert matrix_element4 == matrix_element8
            assert matrix_element5 == matrix_element8
            assert matrix_element6 == matrix_element8
            assert matrix_element7 == matrix_element8

            # now calculate the variance of the QWC decomposition for both the unprocessed Hamiltonian and the processed Hamiltonian, and the tapered Hamiltonian

            if i == j:
                Hqub_tapered_sparse = get_sparse_operator(Hqub_tapered, Nqubits // 2)

                var_og         = variance_of_decomp(qwc_decomp, ket, Nqubits, general=True)
                var_processed  = variance_of_decomp(qwc_decomp_processed, ket, Nqubits, general=True)
                var_tapered    = variance_of_decomp(qwc_decomp_tapered, ket_t, Nqubits // 2, general=True)
                var_lowerbound = variance_of_general_operator(Hqub_tapered_sparse, ket_t).toarray()[0,0]
            else:
                Hqub_tapered_tensor_1q_sparse = get_sparse_operator(
                    Hqub_tapered * (QubitOperator(f'X{Nqubits//2}')+1j*QubitOperator(f'Y{Nqubits//2}')), (Nqubits // 2) + 1
                )

                var_og         = variance_of_decomp(qwc_decomp_swap, bra_ket_composite, Nqubits + 1, general=True)
                var_processed  = variance_of_decomp(qwc_decomp_processed_swap, bra_ket_composite, Nqubits + 1, general=True)
                var_tapered    = variance_of_decomp(qwc_decomp_tapered_swap, bra_ket_composite_t, (Nqubits // 2) + 1, general=True)
                var_lowerbound = variance_of_general_operator(Hqub_tapered_tensor_1q_sparse, bra_ket_composite_t).toarray()[0,0]

            results[(i,j)] = (bra_config, ket_config, var_processed, var_og, var_tapered, var_lowerbound)
            print("{:<2} {:<4} {:<10} {:<15} {:<20} {:<25}".format(
                i, 
                j, 
                np.real(np.round(var_processed, 4)), 
                np.real(np.round(var_og, 4)), 
                np.real(np.round(var_tapered, 4)),
                np.real(np.round(var_lowerbound, 4))
                ))


0  0    0.4666     18.4906         0.4666               0.0006                   
1  0    0.7076     3963.2769       0.4479               0.0124                   
1  1    0.2816     15.5389         0.2816               0.0194                   
2  0    0.7896     3905.2754       0.3138               0.0004                   
2  1    0.7178     3618.551        0.5255               0.0252                   
2  2    0.2798     19.2254         0.2798               0.1631                   
3  0    0.8864     4179.5205       0.5412               0.0002                   
3  1    0.1307     3915.882        0.1464               0.0272                   
3  2    0.1539     3875.5587       0.2005               0.0355                   
3  3    0.5579     31.3941         0.5579               0.3663                   
4  0    0.3691     4129.2761       0.1887               0.0005                   
4  1    0.4585     3836.4445       0.5946               0.0067                   
4  2    0.4094  

In [ ]:
sum([(10**6) * np.real(list(results.values())[i][2]) 
     for i in range(len(results))
     if 0 in list(results.keys())[i] or 1 in list(results.keys())[i] or 2 in list(results.keys())[i] or 3 in list(results.keys())[i] or 4 in list(results.keys())[i]
     ])